# Trajectory Tangling Analysis

Trajectory tangling analysis (Russo et al., 2018 method).

**What this measures:**
- Pre-target: whether tangling vs CTOA follows an inverted-U (quadratic fit)
- Post-target: whether tangling decreases monotonically with CTOA (linear fit)
- Whether pre-target tangling tracks decodable position information
- Whether post-target tangling tracks reaction time

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
from collections import defaultdict
import torch
import matplotlib.pyplot as plt

from src.model import BioLeakyRNN
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    tangling_by_ctoa_bin,
    polynomial_regression,
)
from src.plotting import (
    plot_tangling_timecourses,
    plot_tangling_vs_ctoa,
    plot_tangling_vs_rt,
)

device = "cpu"
print("device:", device)

## 1. Load model and collect trials

In [ ]:
def make_model():
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.05,
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    )


def make_env():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.6, distractor_strength=1.0
    )


model = make_model().to(device)
model.load_state_dict(
    torch.load("checkpoints/stage2.pt", weights_only=True)["state_dict"]
)
model.eval()
print("Loaded stage2 checkpoint.")

In [ ]:
print("Collecting 5000 trials...")
trials = collect_trials(model, make_env, n_trials=5000, device=device)

outcomes = {}
for tr in trials:
    outcomes[tr["train_outcome"]] = outcomes.get(tr["train_outcome"], 0) + 1
print("Outcomes:", outcomes)

## 2. Compute tangling

Windows aligned to target onset:
- **Pre-target**: −300..0 ms (steps −15..0)
- **Post-target**: 0..+600 ms (steps 0..+30)

PCA pre-reduction to 20D before computing tangling (reduces noise, speeds up computation).

In [ ]:
# Pre-target window: -300..0 ms
tang_pre = tangling_by_ctoa_bin(
    trials,
    align_key="target_onset",
    window_before=15,  # 300 ms
    window_after=0,
    pca_dims=20,
    outcome="correct",
    min_trials=10,
)

# Post-target window: 0..+600 ms
tang_post = tangling_by_ctoa_bin(
    trials,
    align_key="target_onset",
    window_before=0,
    window_after=30,  # 600 ms
    pca_dims=20,
    outcome="correct",
    min_trials=10,
)

print(
    f'Pre-target  bins: {tang_pre["labels"]}  Q_mean: {np.round(tang_pre["Q_mean"], 4)}'
)
print(
    f'Post-target bins: {tang_post["labels"]}  Q_mean: {np.round(tang_post["Q_mean"], 4)}'
)

## 3. Tangling time courses per CTOA bin

In [ ]:
# Combined pre + post window in one figure
plot_tangling_timecourses(tang_pre, tang_post, align_label="target_onset")

## 4. Tangling vs CTOA regression

Expected structure:
- Pre-target: **quadratic** fit (inverted-U shape)
- Post-target: **linear** fit (monotone decrease)

In [ ]:
plot_tangling_vs_ctoa(tang_pre, tang_post)

In [ ]:
# Also compare linear vs quadratic for pre-target (AIC comparison)
x = tang_pre["ctoa_ms_mean"]
y = tang_pre["Q_mean"]

for deg in [1, 2]:
    reg = polynomial_regression(x, y, degree=deg)
    n = len(x)
    k = deg + 1  # number of parameters incl. intercept
    ss_res = (
        np.sum((reg["y"] - reg["y_hat"]) ** 2) if reg["y_hat"] is not None else np.nan
    )
    # AIC = n*log(ss_res/n) + 2k
    aic = n * np.log(ss_res / n) + 2 * k if np.isfinite(ss_res) and ss_res > 0 else np.nan
    print(
        f'Pre-target deg-{deg}: R²={reg["r2"]:.3f}  p={reg["p_value"]:.4f}  AIC={aic:.1f}'
    )

## 5. Post-target tangling vs RT

Expectation: longer CTOA → less tangling → faster RT.

In [ ]:
rt_by_bin = defaultdict(list)
for tr in trials:
    if tr["train_outcome"] == "correct" and tr.get("rt_ms") is not None:
        b = tr.get("ctoa_bin")
        if b is not None:
            rt_by_bin[b].append(tr["rt_ms"])

rt_mean = {b: np.mean(v) for b, v in rt_by_bin.items()}
print("Mean RT per CTOA bin (ms):")
for b in sorted(rt_mean):
    print(f"  bin {b}: {rt_mean[b]:.1f} ms  (n={len(rt_by_bin[b])})")

In [ ]:
fig, rho, p = plot_tangling_vs_rt(tang_post, rt_mean)

## 6. RT vs CTOA

Mean reaction time as a function of CTOA (correct trials only).

In [ ]:
# RT vs CTOA — linear regression + error bars
from scipy import stats as sp_stats
import numpy as np

bins = tang_post["labels"]
ctoa_x = tang_post["ctoa_ms_mean"]
rt_y = np.array([np.mean(rt_by_bin[b]) for b in bins])
rt_err = np.array([np.std(rt_by_bin[b], ddof=1) for b in bins])

slope, intercept, r, p_val, _ = sp_stats.linregress(ctoa_x, rt_y)
r2 = r**2
x_fit = np.linspace(ctoa_x.min(), ctoa_x.max(), 200)
y_fit = slope * x_fit + intercept

fig, ax = plt.subplots(figsize=(5, 4))
ax.errorbar(
    ctoa_x,
    rt_y,
    yerr=rt_err,
    fmt="s",
    color="steelblue",
    capsize=4,
    linewidth=1.5,
    markersize=6,
    label="mean +/- SD",
)
ax.plot(
    x_fit,
    y_fit,
    "--",
    color="firebrick",
    linewidth=1.5,
    label="R2=%.2f (p=%.3f)" % (r2, p_val),
)
ax.set_xlabel("CTOA (ms)", fontsize=12)
ax.set_ylabel("Reaction time (ms)", fontsize=12)
ax.set_title("RT vs CTOA", fontsize=12)
ax.legend(fontsize=9)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
print("Linear regression: R2=%.3f  slope=%.4f ms/ms  p=%.4f" % (r2, slope, p_val))

## Summary

| Metric | Model |
|--------|-------|
| Pre-target: quadratic R² | ? |
| Pre-target: quadratic p | ? |
| Post-target: linear R² | ? |
| Post-target: linear p | ? |
| Tangling vs RT: Spearman ρ | ? |
| Tangling vs RT: p | ? |

Fill in '?' from outputs above.

> **Note:** Tangling vs position info requires decoding accuracy per CTOA bin — see `07b_decoding_v2.ipynb`.